In [1]:
import numpy as np

In [2]:
class Piece:
    STRAIGHT = [(1, 0), (-1, 0), (0, 1), (0, -1)]
    DIAGONAL = [(1, 1), (1, -1), (-1, 1), (-1, -1)]
    KNIGHT_OFFSETS = [(1, 2), (2, 1), (-1, 2), (-2, 1),(1, -2), (2, -1), (-1, -2), (-2, -1)]
 
    def __init__(self, color, x_pos, y_pos):
        self.color = color
        self.x_pos = x_pos
        self.y_pos = y_pos
        self.has_moved = False # needed for castle logic

    def move(self, new_x, new_y):
        self.x_pos = new_x
        self.y_pos = new_y
        self.has_moved = True # needed for castle logic

    def get_moves(self, board, en_passant_target=None):  # every subclass must override this
        raise NotImplementedError 

    def display_name(self, end="\n"):
        print(self.color + " " + type(self).__name__, end=end)

    def _slide(self, board, directions):
        moves = []
        for x_change, y_change in directions:
            x, y = self.x_pos + x_change, self.y_pos + y_change
            while 0 <= x < 8 and 0 <= y < 8:
                target = board[y, x]
                if target is None:
                    moves.append((x, y))
                elif target.color != self.color:
                    moves.append((x, y))
                    break
                else:
                    break
                x += x_change
                y += y_change
        return moves

    def _jump(self, board, offsets):
        moves = []
        for x_change, y_change in offsets:
            x, y = self.x_pos + x_change, self.y_pos + y_change
            if 0 <= x < 8 and 0 <= y < 8:
                target = board[y, x]
                if target is None or target.color != self.color:
                    moves.append((x, y))
        return moves


class Rook(Piece):
    def get_moves(self, board, en_passant_target=None):
        return self._slide(board, self.STRAIGHT)


class Bishop(Piece):
    def get_moves(self, board, en_passant_target=None):
        return self._slide(board, self.DIAGONAL)


class Queen(Piece):
    def get_moves(self, board, en_passant_target=None):
        return self._slide(board, self.STRAIGHT + self.DIAGONAL)


class Knight(Piece):
    def get_moves(self, board, en_passant_target=None):
        return self._jump(board, self.KNIGHT_OFFSETS)


class King(Piece):
    def get_moves(self, board, en_passant_target=None):
        return self._jump(board, self.STRAIGHT + self.DIAGONAL)


class Pawn(Piece):
    def get_moves(self, board, en_passant_target=None):
        moves = []
        x, y = self.x_pos, self.y_pos

        # white and black have dif directions
        direction = 1 if self.color == "white" else -1  
        start_row = 1 if self.color == "white" else 6
        
        # Forward one square, and two from the starting row if both squares are empty
        move_one_square = y + direction
        if 0 <= move_one_square < 8 and board[move_one_square, x] is None:
            moves.append((x, move_one_square))
            move_two_squares = y + 2 * direction
            if y == start_row and board[move_two_squares, x] is None:
                moves.append((x, move_two_squares))

        # Diagonal captures (only onto an enemy piece)
        for x_change in (-1, 1):
            capture = x + x_change
            if 0 <= capture < 8 and 0 <= move_one_square < 8:
                target = board[move_one_square, capture]
                if target is not None and target.color != self.color:
                    moves.append((capture, move_one_square))

        # En passant: the target square is the one an enemy pawn just skipped over
        if en_passant_target is not None:
            target_x, target_y = en_passant_target
            if target_y == move_one_square and abs(target_x - x) == 1:
                moves.append(en_passant_target)

        return moves

In [3]:
class Chess:
    #----------------Helpers--------------------
        def is_square_attacked(self, x, y, by_color):
        # TODO: will check whether any by_color piece can reach (x, y)
        return False

    def castling_moves(self, color):
        moves = []
        row = 0 if color == "white" else 7
        enemy = "black" if color == "white" else "white"
        king = self.board[row, 4]

        # king must be on its start square and unmoved
        if not isinstance(king, King) or king.has_moved:
            return moves
            
        # can't castle out of check
        if self.is_square_attacked(4, row, enemy):
            return moves

        # rook column, squares that must be empty, squares the king crosses, king's landing column
        sides = [
            (7, [5, 6],    [5, 6], 6),   # kingside
            (0, [1, 2, 3], [3, 2], 2),   # queenside
        ]

        for rook_x, must_be_empty, king_path, king_landing in sides:
            rook = self.board[row, rook_x]
            if not isinstance(rook, Rook) or rook.has_moved or rook.color != color:
                continue
            if any(self.board[row, x] is not None for x in must_be_empty):
                continue
            if any(self.is_square_attacked(x, row, enemy) for x in king_path):
                continue
            moves.append((king_landing, row))

        return moves
    
    #----------------Code-----------------------
    def __init__(self):
        self.board = np.full((8, 8), None, dtype=object)
        back_row = ["rook", "knight", "bishop", "queen","king", "bishop", "knight", "rook"]
        for x in range(8):
            self.board[0, x] = Piece(back_row[x], "white", x, 0)
            self.board[1, x] = Piece("pawn", "white", x, 1)
            self.board[6, x] = Piece("pawn", "black", x, 6)
            self.board[7, x] = Piece(back_row[x], "black", x, 7)
        
    def display_board(self):
        for i in range(len(self.board)):
            for j in range(len(self.board[0])):
               piece = self.board[i][j]
               if piece is None:
                   print("empty", end=" ")
               else:
                   piece.display_name(end=" ")
            print()
            
    def move(self, old_x, old_y, new_x, new_y, promote_to=None):
        piece = self.board[old_y, old_x]
        target = self.board[new_y, new_x]   # None if the square is empty
        captured = target

        # En passant: a pawn moving diagonally onto an empty square
        if isinstance(piece, Pawn) and new_x != old_x and target is None:
            captured = self.board[old_y, new_x]
            self.board[old_y, new_x] = None

        # Castling: a king moving two columns also moves the rook
        if isinstance(piece, King) and abs(new_x - old_x) == 2:
            if new_x == 6:   # kingside
                rook_from, rook_to = 7, 5
            else:            # queenside
                rook_from, rook_to = 0, 3
            rook = self.board[old_y, rook_from]
            self.board[old_y, rook_to] = rook
            self.board[old_y, rook_from] = None
            rook.move(rook_to, old_y)

        # Move the piece (this also overwrites a normal capture)
        self.board[new_y, new_x] = piece
        self.board[old_y, old_x] = None
        piece.move(new_x, new_y)

        # Promotion: a pawn reaching the last row
        last_row = 7 if piece.color == "white" else 0
        if isinstance(piece, Pawn) and new_y == last_row:
            if promote_to is None:
                #TODO: allow for input to promote to something other than queen
                promote_to = Queen
            new_piece = promote_to(piece.color, new_x, new_y)
            new_piece.has_moved = True
            self.board[new_y, new_x] = new_piece

        # En passant target: set only right after a pawn's double step
        if isinstance(piece, Pawn) and abs(new_y - old_y) == 2:
            self.en_passant_target = (new_x, (old_y + new_y) // 2)
        else:
            self.en_passant_target = None

        return captured
    def play():
        #TODO
        pass
    def winning_eval():
        #TODO
        pass